In [ ]:
import argparse
import tomllib

import sys
sys.path.append("/home/lplaja/crystal_project/src")

import numpy as np
import TBcrystal as cr
import grid as grid
import TBevolution
import Field as Field
import logging
import time
from datetime import datetime

import pprint


filter_type={
    'polygonfilter':grid.polygonfilter
}

evolution_type={
    'TBevolution_CMCP':TBevolution.TBevolution_CMCP,
    'TBevolution_Bloch':TBevolution.TBevolution_Bloch
}   

field_type={
    'polarizedHarmonicElectricField':Field.polarizedHarmonicElectricField,
    'polarizedHarmonicPotentialVectorField':Field.polarizedHarmonicPotentialVectorField
}

field_env_type={
    'env_sin2':Field.env_sin2
}

def load_config(path):
    with open(path, 'rb') as f:
        return tomllib.load(f)
    
config = load_config('/home/lplaja/crystal_project/calculations/graphene_Bloch/config.toml')

ndim=config['crystal']['dim']
a=config['crystal']['a']
W90filename=config['crystal']['W90filename']

if ndim==1:
    limits=[config['BZ']['kx_min'],config['BZ']['kx_max']]  
    nk=[config['BZ']['nkx']]
elif ndim==2:
    limits=[[config['BZ']['kx_min'],config['BZ']['kx_max']],[config['BZ']['ky_min'],config['BZ']['ky_max']]]  
    nk=[config['BZ']['nkx'],config['BZ']['nky']]
else:
    limits=[[config['BZ']['kx_min'],config['BZ']['kx_max']],[config['BZ']['ky_min'],config['BZ']['ky_max']],[config['BZ']['kz_min'],config['BZ']['kz_max']]]  
    nk=[config['BZ']['nkx'],config['BZ']['nky'],config['BZ']['nkz']]

if 'filter' in config['BZ']:
    if config['BZ']['filter']['type']=='polygonfilter':
        filter=filter_type[config['BZ']['filter']['type']]
        filter_args=config['BZ']['filter']['args']
    else:
        filter=None
        filter_args={}
else:
    filter=None
    filter_args={}

gr=grid.UniformCartesianGrid(ndim,limits,nk, origin=(0.0,0), filter=filter, filter_args= filter_args)

logging.info('Constructing the TB crystal')

filename=config['crystal']['W90filename']
species_name=config['crystal']['name']
threshold_hopping=config['crystal']['threshold_hopping']

TBcr=cr.crystal.from_W90_TB_file(filename=filename , 
                                grid=gr, species_name=species_name, threshold_hopping=threshold_hopping)

print(f'vprint in line: 109 --> {filename=}')
print(f'vprint in line: 109 --> {species_name=}')
print(f'vprint in line: 109 --> {threshold_hopping=}')

logging.info('Constructing the temporal grid')

lambda0=config['Field']['lambda']*1e-9
T0=Field.lambda2T(lambda0)
tini=config['time']['tini']*T0
tfin=config['time']['tfin']*T0
limits=[tini,tfin]
nptx=config['time']['nt']
tt=grid.UniformCartesianGrid(1,limits=limits, nptx=nptx)

print(f'vprint in line: 120 --> {lambda0=}')
print(f'vprint in line: 120 --> {T0=}')
print(f'vprint in line: 120 --> {tini=}')
print(f'vprint in line: 120 --> {tfin=}')
print(f'vprint in line: 120 --> {nptx=}')

logging.info('Constructing the Field')

fieldcallable=field_type[config['Field']['type']]    
I_W__cm2=config['Field']['I_W__cm2']
env=field_env_type[config['Field']['env']['type']]
env_params=config['Field']['env']['args']
phi_rad=config['Field']['phi']
chi_rad=config['Field']['chi']
ellip=config['Field']['ellip']

Efield=fieldcallable(tt,I_W__cm2=I_W__cm2,lambda0_nm=lambda0*1e9, env=env, env_parameters=env_params, phi_rad=phi_rad, chi_rad=chi_rad,ellip=ellip)
print(f'vprint in line: 143 --> {I_W__cm2=}')
print(f'vprint in line: 143 --> {env=}')
print(f'vprint in line: 143 --> {env_params=}')
print(f'vprint in line: 143 --> {phi_rad=}')
print(f'vprint in line: 143 --> {chi_rad=}')
print(f'vprint in line: 143 --> {ellip=}')
print(f'vprint in line: 46 --> {max(Efield.E[:,0])=}')
print(f'vprint in line: 46 --> {max(Efield.E[:,1])=}')


logging.info('Constructing the TB evolver')

evolcallable=evolution_type[config['evolver']['type']]
TBev=evolcallable(TBcr, Efield)

vprint in line: 97 --> ndim=2
vprint in line: 97 --> limits=[[-0.3048780488, 0.3048780488], [-0.3048780488, 0.3048780488]]
vprint in line: 97 --> nk=[500, 500]
vprint in line: 97 --> filter=<function polygonfilter at 0x7976903bae80>
vprint in line: 97 --> filter_args={'nsides': 6, 'radius': 0.27100271}
vprint in line: 273 --> deltas=array([[0.       , 0.       , 0.       ],
       [1.4202816, 0.       , 0.       ]])
vprint in line: 109 --> filename='../Wannier90 data/gr1NN_tb.dat'
vprint in line: 109 --> species_name='graphene'
vprint in line: 109 --> threshold_hopping=0.1

TBcr.direct_lattice_unit=1e-10
TBcr.direct_vectors=array([[ 2.13042249,  1.23      ,  0.        ],
       [ 2.13042249, -1.23      ,  0.        ],
       [ 0.        ,  0.        , 15.        ]])
TBcr.orbital_names=['A', 'B']
TBcr.deg_weights=array([1, 1, 1, 1, 1])
TBcr.R_vectors=array([[-1,  0,  0],
       [ 0, -1,  0],
       [ 0,  0,  0],
       [ 0,  1,  0],
       [ 1,  0,  0]], dtype=int32)
TBcr.h_Rnm=array([[